<a href="https://colab.research.google.com/github/chethana-152005/Yuvaintern-Logistics-Data-Analyst-Intern/blob/main/task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Load data
df = pd.read_csv('Food_Delivery_Time_Prediction.csv')

# 1. Drop non-predictive text columns
df = df.drop(['Order_ID', 'Order_Date'], axis=1)

# 2. Automatically One-Hot Encode ALL remaining categorical (text) columns
# This guarantees no strings are left behind to cause the ValueError
df_encoded = pd.get_dummies(df, drop_first=True)

# Define Features (X) and Target (y)
X = df_encoded.drop('Time_taken_min', axis=1)
y = df_encoded['Time_taken_min']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Baseline Model (Linear Regression)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Train Ensemble Model (Random Forest)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

# Evaluation Function
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} Performance ---")
    print(f"MAE: {mae:.2f} minutes")
    print(f"RMSE: {rmse:.2f} minutes")
    print(f"R-squared: {r2:.2f}\n")

evaluate_model(y_test, lr_preds, "Linear Regression")
evaluate_model(y_test, rf_preds, "Random Forest Regressor")

# Feature Importance Analysis for Optimization
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
print("Top 5 Driving Factors of Delivery Time:")
print(feat_df.sort_values(by='Importance', ascending=False).head(5))

--- Linear Regression Performance ---
MAE: 6.37 minutes
RMSE: 9.22 minutes
R-squared: 0.93

--- Random Forest Regressor Performance ---
MAE: 2.96 minutes
RMSE: 3.82 minutes
R-squared: 0.99

Top 5 Driving Factors of Delivery Time:
                   Feature  Importance
8         Road_Distance_km    0.514506
10      Average_Speed_kmph    0.430663
7     Preparation_Time_Min    0.053408
9        Number_of_Signals    0.000504
3   Rider_Experience_Years    0.000128
